[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CharlesShang/TorchCode/blob/master/solutions/58_multi_head_latent_attention_solution.ipynb)

# 🔴 Solution: Multi-Head Latent Attention

Reference solution for `multi_head_latent_attention`.

In [ ]:
# Install the latest torch-judge from this repo in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q --force-reinstall --no-deps git+https://github.com/CharlesShang/TorchCode.git@master')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import math


In [ ]:
# ✅ SOLUTION

class MultiHeadLatentAttention(nn.Module):
    def __init__(self, dim: int, num_heads: int, latent_dim: int):
        super().__init__()
        assert dim % num_heads == 0
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.latent_dim = latent_dim
        self.q_proj = nn.Linear(dim, dim)
        self.kv_down = nn.Linear(dim, latent_dim)
        self.k_up = nn.Linear(latent_dim, dim)
        self.v_up = nn.Linear(latent_dim, dim)
        self.out_proj = nn.Linear(dim, dim)

    def _heads(self, x):
        B, S, D = x.shape
        return x.view(B, S, self.num_heads, self.head_dim).transpose(1, 2)

    def forward(self, x: torch.Tensor, latent_cache: torch.Tensor | None = None):
        B, S, D = x.shape
        current_latent = self.kv_down(x)
        if latent_cache is None:
            all_latent = current_latent
        else:
            all_latent = torch.cat([latent_cache, current_latent], dim=1)
        q = self._heads(self.q_proj(x))
        k = self._heads(self.k_up(all_latent))
        v = self._heads(self.v_up(all_latent))
        attn = torch.softmax((q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim), dim=-1)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, S, D)
        return self.out_proj(out), all_latent


In [ ]:
# Verify
m = MultiHeadLatentAttention(dim=16, num_heads=4, latent_dim=6)
x = torch.randn(2, 3, 16)
out, cache = m(x)
print(out.shape, cache.shape)
out2, cache2 = m(torch.randn(2, 1, 16), cache)
print(out2.shape, cache2.shape)


In [ ]:
# Run judge
from torch_judge import check
check('multi_head_latent_attention')
